# Quick Model Diagnostic

A lightweight version of the testing framework to quickly check feasibility and diagnostics.

In [12]:
import numpy as np
import pandas as pd
import time
import cvxpy as cp

from cma.data_reader import (
    read_vessel_class_data,
    read_port_data,
    read_sailing_distance_data,
    read_demand_with_transit_time,
    read_cnc_proforma_data,
)
from cma.port import PortGraph
from cma.servicegraph import ServiceGraph

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. Load Data (Minimal Subset)

In [13]:
vesselpool = read_vessel_class_data()
portpool_main, portpool_dmd = read_port_data()
dist_matrix = read_sailing_distance_data(portpool_main)
demand_matrix, transit_time_matrix = read_demand_with_transit_time(portpool_main)

portgraph = PortGraph(
    portpool_main, 
    dist_matrix, 
    demand_matrix,
    mat_transit_time=transit_time_matrix,
    filter_by_demand=False
)

proforma = read_cnc_proforma_data(portpool_main, vesselpool)
all_service_lines = proforma['lines']

# SUBSET FOR SPEED
service_lines = all_service_lines[:10]
servicegraph = ServiceGraph(service_lines)

trans_ports = portgraph.filtered_by_transship_capacity()
od_pairs_dict = servicegraph.get_all_paths(portgraph, trans_ports)

all_od_pairs = od_pairs_dict['od_pairs']
all_demands = od_pairs_dict['od_pairs_demand']
all_paths = od_pairs_dict['od_pairs_path']

filtered_pairs = []
filtered_paths = []
for od, paths, dmd in zip(all_od_pairs, all_paths, all_demands):
    if dmd > 0:
        filtered_pairs.append(od)
        filtered_paths.append(paths)

# SUBSET OD PAIRS
od_pairs = filtered_pairs[:50]
od_paths = filtered_paths[:50]

print(f"Loaded {len(service_lines)} lines and {len(od_pairs)} OD pairs for quick test.")

Loaded 10 lines and 50 OD pairs for quick test.


## 2. Run Optimization (All Features)

In [14]:
tuneparams = {
    'turnon-transship_shipclass_restriction': 1,
    'turnon-vessel_speed_optimization': 0,       # 0=ON (accurate)
    'turnon-port_operations_constraint': 1,
    'turnon-transit_time_penalty': 1,
    'ctrparam-kts_buffer': 0,
    'ctrparam-transship_A': 100,
    'ctrparam-speed_soft_cap_kts': 16.5,
    'ctrparam-speed_penalty_multiplier': 2.0,
    'ctrparam-transit_penalty_multiplier': 1000.0,
    'ctrparam-buffer_penalty_below_15pct': 1000.0,
    'ctrparam-buffer_penalty_above_30pct': 2000.0,
    'unfulfilled_demand_penalty': 1e6,
    'BigM-transship': 10000,
    'BigM-n_ships': 10,
    'BigM-saildays': 100,
    'BigM-line_capacity': 30000,
    'BigM-portcall_cost': 1e7,          
    'turnon-schedule_adherence': 0,     # Tether optimized arrival times to proforma
    'schedule_buffer_hrs': 120.0,        # 120 hour buffer for tethering
    'solver-MIPGap': 0.10,              # Experimental: tolerate 10% optimality gap
    'solver-TimeLimit': 1000,           # Experimental: roughly 16 minutes
    'solver-MIPFocus': 1,               # Focus on finding feasible solutions quickly
    'solver-verbose': True              # Show Gurobi's progress logs
}

week_levels = [1, 2, 3, 4, 5, 6, 7]

## 4. Full Dataset Evaluation

Testing the model on all service lines and all positive demand OD pairs to identify potential bottlenecks.

In [15]:
service_lines_full = all_service_lines
servicegraph_full = ServiceGraph(service_lines_full)

print("Finding all paths for full dataset...")
trans_ports = portgraph.filtered_by_transship_capacity()
od_pairs_dict_full = servicegraph_full.get_all_paths(portgraph, trans_ports)

all_od_pairs_full = od_pairs_dict_full['od_pairs']
all_demands_full = od_pairs_dict_full['od_pairs_demand']
all_paths_full = od_pairs_dict_full['od_pairs_path']

filtered_pairs_full = []
filtered_paths_full = []
for od, paths, dmd in zip(all_od_pairs_full, all_paths_full, all_demands_full):
    if dmd > 0:
        filtered_pairs_full.append(od)
        filtered_paths_full.append(paths)

print(f"Full Dataset: {len(service_lines_full)} lines and {len(filtered_pairs_full)} OD pairs.")

Finding all paths for full dataset...
Full Dataset: 31 lines and 741 OD pairs.


In [ ]:
full_week_levels = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
full_tuneparams = tuneparams.copy()
full_tuneparams.update({
    # Match the successful grid-search profile.
    'unfulfilled_demand_penalty': 1e6,
    'BigM-line_capacity': 30000,
    'solver-MIPGap': 0.10,
    'solver-TimeLimit': 1000,
    'solver-MIPFocus': 1,
    'solver-verbose': False,
})

start_time = time.time()
solution_full = servicegraph_full.fulfill_demands(
    filtered_pairs_full,
    filtered_paths_full,
    portgraph,
    vesselpool,
    full_week_levels,
    full_tuneparams
)
end_time = time.time()

print(f"Full Solve Time: {end_time - start_time:.2f}s")
print(f"Total Cost: {solution_full['total cost']:,.2f}")
print(f"  - Chartering Cost:    {solution_full.get('chartering cost', 0):,.2f}")
print(f"  - Transshipment Cost: {solution_full.get('transshipment cost', 0):,.2f}")
print(f"  - Bunkering Cost:     {solution_full.get('bunkering cost', 0):,.2f}")
print(f"  - Port Call Cost:     {solution_full.get('portcall cost', 0):,.2f}")

print(f"\n--- Solver Diagnostics ---")
print(f"Solver Status:   {solution_full.get('solver_status')}")
print(f"Solver Name:     {solution_full.get('solver_name')}")
print(f"Solver Time:     {solution_full.get('solver_solve_time')}")
print(f"MIP Gap:         {solution_full.get('solver_mip_gap')}")
print(f"Best Bound:      {solution_full.get('solver_best_bound')}")
print(f"Solver Obj:      {solution_full.get('solver_obj_val')}")
print(f"Node Count:      {solution_full.get('solver_node_count')}")

print(f"\n--- KPI Metrics ---")
teus_in = solution_full.get('kpi_teus_input', 0)
teus_out = solution_full.get('kpi_teus_fulfilled', 0)
teus_del = solution_full.get('kpi_teus_delayed', 0)
pct_fulfilled = (teus_out / teus_in * 100) if teus_in > 0 else 0
pct_delayed = (teus_del / teus_out * 100) if teus_out > 0 else 0

print(f"TEUs Input:     {teus_in:,.0f}")
print(f"TEUs Fulfilled: {teus_out:,.0f} ({pct_fulfilled:.1f}%)")
print(f"TEUs Delayed:   {teus_del:,.0f} ({pct_delayed:.1f}%)")
print(f"Avg Delays:     {solution_full.get('kpi_avg_delay_days', 0):.2f} days")
print(f"Buffer > 30%:   {solution_full.get('kpi_lines_buffer_above_30', 0)} lines")
print(f"Buffer < 15%:   {solution_full.get('kpi_lines_buffer_below_15', 0)} lines")

c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 280 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 281 times so far.

  warnings.warn(msg, UserWarning)
c:

In [ ]:
if solution_full['total cost'] == float('inf'):
    print("\n❌ FULL DATASET INFEASIBLE. Identifying problematic components...")
    
    # 1. Check for ports with zero productivity that have demand
    zero_prod_ports = []
    for port in portgraph.tolist_port():
        prods = port.get_producticity(vesselpool)
        if sum(prods) == 0:
            zero_prod_ports.append(port.get_id())
    
    if zero_prod_ports:
        print(f"\nPorts with ZERO productivity: {zero_prod_ports}")
        # Check if any OD pair involves these ports
        for i, (o_idx, d_idx) in enumerate(filtered_pairs_full):
            o_id = portgraph.get_port_by_idx(o_idx).get_id()
            d_id = portgraph.get_port_by_idx(d_idx).get_id()
            if o_id in zero_prod_ports or d_id in zero_prod_ports:
                print(f"  ⚠️ OD Pair {o_id}->{d_id} involves zero-productivity port.")

    # 2. Check for impossible distance/speed requirements (Hard Constraints)
    print("\nChecking for distance/speed violations:")
    for line in service_lines_full:
        dist = line.get_distance(portgraph)
        max_weeks = max(week_levels)
        min_speed_needed = dist / (24 * (7 * max_weeks - 1.0)) # 1 day min stay
        if min_speed_needed > 18.5:
            print(f"  ❌ Line {line.name()}: {dist:.0f}nm needs {min_speed_needed:.1f} kts at {max_weeks} weeks (Max 18.5)")

else:
    print("\n✓ FULL DATASET FEASIBLE!")
    
    # Violation Summary
    violation_lb = solution_full['buffer violation lb']
    violation_ub = solution_full['buffer violation ub']
    violation_summary = []
    for i, line in enumerate(service_lines_full):
        lb = violation_lb[i].value if hasattr(violation_lb[i], 'value') else 0
        ub = violation_ub[i].value if hasattr(violation_ub[i], 'value') else 0
        if lb > 0.1 or ub > 0.1:
            violation_summary.append({'Line': line.name(), 'LB_Violation': lb, 'UB_Violation': ub})
    
    if violation_summary:
        print("\nSignificant Buffer Violations in Full Dataset:")
        print(pd.DataFrame(violation_summary))


✓ FULL DATASET FEASIBLE!

Significant Buffer Violations in Full Dataset:
       Line  LB_Violation  UB_Violation
0   BBX3CNC      0.000000     84.663661
1    BBXCNC      0.000000      2.321186
2    BMXCNC      0.000000     80.303747
3    CP2CNC      0.000000      6.347184
4    CP3CNC      0.000000      2.359413
5    CP8CNC      0.000000      4.042614
6    CS1CNC     16.992492      0.000000
7    CV8CNC      0.000000      4.472209
8    KCSCNC      8.600000      0.000000
9   SGS2CNC      0.000000      4.199653
10   SGSCNC      0.000000      0.199653
11   YCXCNC      0.000000     14.057139
12   YSXCNC      0.000000      7.200149


## 5. Speed-Simplified Model (Full Dataset)

Running the model where vessel speed optimization is simplified (turnon-vessel_speed_optimization > 1/2).

In [ ]:
tuneparams_simple = tuneparams.copy()
tuneparams_simple['turnon-vessel_speed_optimization'] = 1  # 1 = OFF (simplified)

start_time = time.time()
solution_simple = servicegraph_full.fulfill_demands(
    filtered_pairs_full,
    filtered_paths_full,
    portgraph,
    vesselpool,
    week_levels,
    tuneparams_simple
)
end_time = time.time()

print(f"Simple Model Solve Time: {end_time - start_time:.2f}s")
print(f"Total Cost: {solution_simple['total cost']:,.2f}")
print(f"  - Chartering Cost:    {solution_simple.get('chartering cost', 0):,.2f}")
print(f"  - Transshipment Cost: {solution_simple.get('transshipment cost', 0):,.2f}")
print(f"  - Bunkering Cost:     {solution_simple.get('bunkering cost', 0):,.2f}")
print(f"  - Port Call Cost:     {solution_simple.get('portcall cost', 0):,.2f}")

print(f"\n--- Solver Diagnostics ---")
print(f"Solver Status:   {solution_simple.get('solver_status')}")
print(f"Solver Name:     {solution_simple.get('solver_name')}")
print(f"Solver Time:     {solution_simple.get('solver_solve_time')}")
print(f"MIP Gap:         {solution_simple.get('solver_mip_gap')}")
print(f"Best Bound:      {solution_simple.get('solver_best_bound')}")
print(f"Solver Obj:      {solution_simple.get('solver_obj_val')}")
print(f"Node Count:      {solution_simple.get('solver_node_count')}")

print(f"\n--- KPI Metrics ---")
teus_in = solution_simple.get('kpi_teus_input', 0)
teus_out = solution_simple.get('kpi_teus_fulfilled', 0)
teus_del = solution_simple.get('kpi_teus_delayed', 0)
pct_fulfilled = (teus_out / teus_in * 100) if teus_in > 0 else 0
pct_delayed = (teus_del / teus_out * 100) if teus_out > 0 else 0

print(f"TEUs Input:     {teus_in:,.0f}")
print(f"TEUs Fulfilled: {teus_out:,.0f} ({pct_fulfilled:.1f}%)")
print(f"TEUs Delayed:   {teus_del:,.0f} ({pct_delayed:.1f}%)")
print(f"Avg Delays:     {solution_simple.get('kpi_avg_delay_days', 0):.2f} days")
print(f"Buffer > 30%:   {solution_simple.get('kpi_lines_buffer_above_30', 0)} lines")
print(f"Buffer < 15%:   {solution_simple.get('kpi_lines_buffer_below_15', 0)} lines")

c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 32 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 33 times so far.

  warnings.warn(msg, UserWarning)
c:\U

                                     CVXPY                                     
                                     v1.7.5                                    


(CVXPY) May 08 12:33:01 AM: Your problem has 25082 variables, 21349 constraints, and 0 parameters.
(CVXPY) May 08 12:33:04 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) May 08 12:33:04 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) May 08 12:33:04 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) May 08 12:33:04 AM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) May 08 12:33:09 AM: Compiling problem (target solver=GUROBI).
(CVXPY) May 08 12:33:09 AM: Reduction chain: CvxAttr2Constr -> Qp2SymbolicQp -> QpMatrixStuffing -> GUROBI
(CVXPY) May 08 12:33:09 AM: Applying reduction CvxAttr2Constr


-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) May 08 12:33:15 AM: Applying reduction Qp2SymbolicQp
(CVXPY) May 08 12:33:27 AM: Applying reduction QpMatrixStuffing
(CVXPY) May 08 12:38:40 AM: Applying reduction GUROBI
(CVXPY) May 08 12:38:41 AM: Finished problem compilation (took 3.361e+02 seconds).
(CVXPY) May 08 12:38:41 AM: Invoking solver GUROBI  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
Set parameter OutputFlag to value 1
Set parameter QCPDual to value 1
Set parameter MIPGap to value 0.1
Set parameter TimeLimit to value 1000
Set parameter MIPFocus to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  1000
MIPGap  0.1
MIPFocus  1
QCPDual  1

Optimize a model with 21349 rows, 25082 columns and 312384 nonzeros
Model fingerprint: 0xa0a18539
Variable types: 18882 continuous, 6200 integer (5859 binary)
Coefficient statistics:
  Matrix range     [4e-05, 1e+07]
  Objective range  [1e-01, 1e+06]
  Bounds ran

(CVXPY) May 08 12:39:18 AM: Problem status: optimal
(CVXPY) May 08 12:39:18 AM: Optimal value: 5.500e+07
(CVXPY) May 08 12:39:18 AM: Compilation took 3.361e+02 seconds
(CVXPY) May 08 12:39:18 AM: Solver (including time spent in interface) took 3.662e+01 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Simple Model Solve Time: 397.26s
Total Cost: 55,000,717.28
  - Chartering Cost:    6,630,601.98
  - Transshipment Cost: 473,483.62
  - Bunkering Cost:     6,838,988.51
  - Port Call Cost:     1,405,591.13

--- Solver Diagnostics ---
Solver Status:   optimal
Solver Name:     GUROBI
Solver Time:     36.16099977493286
MIP Gap:         0.06844629532905946
Best Bound:      51236121.945425145
Solver Obj:      55000717.283952676
Node Count:      1251.0

--- KPI Metrics ---
TEUs Input:     70,850
TEUs Fulfilled: 70,850 (100.0%)
TEUs Delayed:   4,948 (7.0%)
Avg Delays:     8.01 days
Buffer > 30%:   0 lines
Buffer < 15%:   2 lines


In [ ]:
if solution_simple['total cost'] == float('inf'):
    print("\n❌ SIMPLE MODEL INFEASIBLE.")
else:
    print("\n✓ SIMPLE MODEL FEASIBLE!")
    
    # Violation Summary
    violation_lb = solution_simple['buffer violation lb']
    violation_ub = solution_simple['buffer violation ub']
    violation_summary = []
    for i, line in enumerate(service_lines_full):
        lb = violation_lb[i].value if hasattr(violation_lb[i], 'value') else 0
        ub = violation_ub[i].value if hasattr(violation_ub[i], 'value') else 0
        if lb > 0.1 or ub > 0.1:
            violation_summary.append({'Line': line.name(), 'LB_Violation': lb, 'UB_Violation': ub})
    
    if violation_summary:
        print("\nSignificant Buffer Violations in Simple Model:")
        print(pd.DataFrame(violation_summary))


✓ SIMPLE MODEL FEASIBLE!

Significant Buffer Violations in Simple Model:
     Line  LB_Violation  UB_Violation
0  CS1CNC     29.161157           0.0
1  KCSCNC      3.014740           0.0


## 6. Schedule Adherence & Transshipment Analysis

Analyze how closely optimized schedules follow the proforma tether, and examine transshipment wait times (enforcing the 1-day minimum).

In [ ]:
print("--- Schedule Adherence (Tethering) Analysis ---")
if solution_simple['total cost'] != float('inf'):
    stay_days = solution_simple['port staying days'].value
    weeks_vars = solution_simple['weeks']
    
    adherence_data = []
    for i, line in enumerate(service_lines_full):
        proforma_sched = line.get_schedule(portgraph, vesselpool)
        if not proforma_sched: continue
        
        # Get picked week level
        wk_picked = 0
        for j, wk in enumerate(week_levels):
            if weeks_vars[i, j].value > 0.5: 
                wk_picked = wk
                break
        
        line_port_stay_days = np.sum(stay_days[i, :])
        line_sailing_days = 7 * wk_picked - line_port_stay_days
        
        anchor_wd, anchor_hr = line.get_anchor_eosp()
        base_eosp_days = (anchor_wd * 24 + anchor_hr) / 24.0
        total_dist = line.get_distance(portgraph)
        
        cum_dist = 0
        cum_stay_days = 0
        line_ports = line.tolist_port()
        
        for k, port in enumerate(line_ports):
            p_idx = portgraph.get_unique_index(port)
            num_visits = line_ports.count(port)
            
            if k > 0:
                cum_dist += portgraph.get_distance(line_ports[k-1], port)
            
            # Optimized ETB (days from Mon 00:00)
            opt_etb = base_eosp_days + (cum_dist / total_dist) * line_sailing_days + cum_stay_days
            prof_etb = proforma_sched[k][0] / 24.0
            
            # Deviation in hours
            deviation = (opt_etb - prof_etb) * 24.0
            
            adherence_data.append({
                'Line': line.name(),
                'Port': port.get_id(),
                'Opt ETB (h)': opt_etb * 24 % 168,
                'Prof ETB (h)': prof_etb * 24 % 168,
                'Dev (h)': deviation
            })
            
            cum_stay_days += stay_days[i, p_idx] / num_visits
            
    df_adherence = pd.DataFrame(adherence_data)
    print(f"\nAverage Schedule Deviation: {df_adherence['Dev (h)'].mean():.2f} hours")
    print(f"Max Schedule Deviation: {df_adherence['Dev (h)'].max():.2f} hours")
    
    print("\nLines with largest deviations:")
    print(df_adherence.sort_values('Dev (h)', ascending=False).head(10))

print("\n--- Transshipment Wait Time Analysis (Enforcing 1-Day Min) ---")
if solution_simple['total cost'] != float('inf'):
    ts_wait_data = []
    line_schedules = [line.get_schedule(portgraph, vesselpool) for line in service_lines_full]
    
    # Sample some transshipments from the paths
    count = 0
    for od_idx, paths in enumerate(filtered_paths_full):
        for path in paths:
            slots = path.tolist_slot()
            for i in range(len(slots) - 1):
                s1, s2 = slots[i], slots[i+1]
                if s1.get_service_name() != s2.get_service_name():
                    # Transshipment!
                    l1_idx = service_lines_full.index(s1.get_service())
                    l2_idx = service_lines_full.index(s2.get_service())
                    
                    seg1_idx = s1.get_service().get_segment_idx(s1.get_segment())
                    seg2_idx = s2.get_service().get_segment_idx(s2.get_segment())
                    
                    etd1 = line_schedules[l1_idx][seg1_idx][1]
                    etb2 = line_schedules[l2_idx][seg2_idx][0]
                    
                    wait_hrs = (etb2 - etd1) % 168.0
                    is_one_day_penalty = wait_hrs < 24.0
                    final_wait = wait_hrs + 168.0 if is_one_day_penalty else wait_hrs
                    
                    ts_wait_data.append({
                        'Hub': s1.get_end().get_id(),
                        'From': s1.get_service_name(),
                        'To': s2.get_service_name(),
                        'Raw Wait (h)': wait_hrs,
                        'Penalty Applied': is_one_day_penalty,
                        'Total Wait (h)': final_wait
                    })
                    count += 1
        if count > 50: break # Just a sample
        
    if ts_wait_data:
        print(pd.DataFrame(ts_wait_data).drop_duplicates().head(20))
    else:
        print("No transshipments found in sample paths.")

--- Schedule Adherence (Tethering) Analysis ---

Average Schedule Deviation: -102.97 hours
Max Schedule Deviation: 77.25 hours

Lines with largest deviations:
        Line   Port  Opt ETB (h)  Prof ETB (h)    Dev (h)
153   NPFCNC  JPHKT   162.645997          85.4  77.245997
154   NPFCNC  KRKAN    14.271639         109.4  72.871639
150   NPFCNC  JPSBS    22.348167          20.0   2.348167
151   NPFCNC  JPOIT    48.958760          51.0  -2.041240
103  JTVSCNC  JPTYO    80.000000          82.9  -2.900000
42    CP2CNC  CNSHK   134.000000         138.0  -4.000000
65    CS2CNC  CNSHK   152.000000         156.2  -4.200000
164   SP8CNC  PHDVO    88.905647          93.6  -4.694353
94   JPXSCNC  JPTYO   151.000000         156.2  -5.200000
114   JTXCNC  JPTYO    21.000000          26.3  -5.300000

--- Transshipment Wait Time Analysis (Enforcing 1-Day Min) ---
      Hub     From       To  Raw Wait (h)  Penalty Applied  Total Wait (h)
0   MYPKG   YCXCNC  JPXSCNC          62.0            False      

C:\Users\ASUS\AppData\Local\Temp\ipykernel_23268\3203742648.py:37: RuntimeWarning: invalid value encountered in scalar divide
  opt_etb = base_eosp_days + (cum_dist / total_dist) * line_sailing_days + cum_stay_days


In [ ]:
# Check unique ports in the full dataset using the available methods
if 'servicegraph_full' in locals():
    all_ports = set()
    # Iterate through all service lines in the graph
    for line in servicegraph_full.tolist_serviceLine():
        # Get the list of Port objects for this line
        for port in line.tolist_port():
            all_ports.add(port.get_id())
    
    print(f"Total unique ports in the FULL dataset (31 lines): {len(all_ports)}")
    # Optional: print the list of ports
    # print(sorted(list(all_ports)))

elif 'service_lines_full' in locals():
    # Fallback to the raw list of service lines if the graph wasn't built
    all_ports = set()
    for line in service_lines_full:
        for port in line.tolist_port():
            all_ports.add(port.get_id())
    print(f"Total unique ports in service_lines_full: {len(all_ports)}")

else:
    print("Variables 'servicegraph_full' or 'service_lines_full' not found. Please run the data loading cells.")

Total unique ports in the FULL dataset (31 lines): 56


In [ ]:
# --- Standalone grid search: unfulfilled demand penalty vs fulfillment ---
# This cell is self-contained: run it directly without executing previous cells.

import time
import numpy as np
import pandas as pd

from cma.data_reader import (
    read_vessel_class_data,
    read_port_data,
    read_sailing_distance_data,
    read_demand_with_transit_time,
    read_cnc_proforma_data,
)
from cma.port import PortGraph
from cma.servicegraph import ServiceGraph

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

# 1) Rebuild full dataset context
print("Loading full dataset context...")
vesselpool = read_vessel_class_data()
portpool_main, _ = read_port_data()
dist_matrix = read_sailing_distance_data(portpool_main)
demand_matrix, transit_time_matrix = read_demand_with_transit_time(portpool_main)

portgraph = PortGraph(
    portpool_main,
    dist_matrix,
    demand_matrix,
    mat_transit_time=transit_time_matrix,
    filter_by_demand=False,
)

proforma = read_cnc_proforma_data(portpool_main, vesselpool)
service_lines_full = proforma['lines']
servicegraph_full = ServiceGraph(service_lines_full)

trans_ports = portgraph.filtered_by_transship_capacity()
od_pairs_dict_full = servicegraph_full.get_all_paths(portgraph, trans_ports)

all_od_pairs_full = od_pairs_dict_full['od_pairs']
all_demands_full = od_pairs_dict_full['od_pairs_demand']
all_paths_full = od_pairs_dict_full['od_pairs_path']

filtered_pairs_full = []
filtered_paths_full = []
for od, paths, dmd in zip(all_od_pairs_full, all_paths_full, all_demands_full):
    if dmd > 0:
        filtered_pairs_full.append(od)
        filtered_paths_full.append(paths)

print(f"Prepared full problem: {len(service_lines_full)} lines, {len(filtered_pairs_full)} positive-demand OD pairs")

# 2) Base params (complete model: speed optimization ON when set to 0)
week_levels = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
base_tuneparams = {
    'turnon-transship_shipclass_restriction': 1,
    'turnon-vessel_speed_optimization': 0,      # 0 = complete/accurate speed model
    'turnon-port_operations_constraint': 1,
    'turnon-transit_time_penalty': 1,
    'ctrparam-kts_buffer': 0,
    'ctrparam-transship_A': 100,
    'ctrparam-speed_soft_cap_kts': 16.5,
    'ctrparam-speed_penalty_multiplier': 2.0,
    'ctrparam-transit_penalty_multiplier': 1000.0,
    'ctrparam-buffer_penalty_below_15pct': 1000.0,
    'ctrparam-buffer_penalty_above_30pct': 2000.0,
    'BigM-transship': 10000,
    'BigM-n_ships': 10,
    'BigM-saildays': 100,
    'BigM-line_capacity': 30000,
    'BigM-portcall_cost': 1e7,
    'turnon-schedule_adherence': 0,
    'schedule_buffer_hrs': 120.0,
    # Keep limits modest for grid-search practicality; increase if needed
    'solver-MIPGap': 0.10,
    'solver-TimeLimit': 1000,
    'solver-MIPFocus': 1,
    'solver-verbose': False,
}

# 3) Grid to test (coarse -> can refine around best)
penalty_grid = [
    1e6,
    3e6,
    1e7,
    3e7,
    1e8,
    3e8,
    1e9,
]

print("\nStarting grid search on unfulfilled_demand_penalty...")
rows = []

for penalty in penalty_grid:
    tuneparams = dict(base_tuneparams)
    tuneparams['unfulfilled_demand_penalty'] = float(penalty)

    t0 = time.time()
    try:
        sol = servicegraph_full.fulfill_demands(
            filtered_pairs_full,
            filtered_paths_full,
            portgraph,
            vesselpool,
            week_levels,
            tuneparams,
        )
        elapsed = time.time() - t0

        total_cost = sol.get('total cost', float('inf'))
        teus_in = float(sol.get('kpi_teus_input', 0.0))
        teus_fulfilled = float(sol.get('kpi_teus_fulfilled', 0.0))
        teus_delayed = float(sol.get('kpi_teus_delayed', 0.0))
        avg_delay_days = float(sol.get('kpi_avg_delay_days', 0.0))
        solver_mip_gap = sol.get('solver_mip_gap')
        solver_best_bound = sol.get('solver_best_bound')
        solver_obj = sol.get('solver_obj_val')
        solver_status = sol.get('solver_status')
        solver_time = sol.get('solver_solve_time')
        solver_nodes = sol.get('solver_node_count')

        fulfillment_pct = (100.0 * teus_fulfilled / teus_in) if teus_in > 0 else 0.0
        delayed_pct = (100.0 * teus_delayed / teus_fulfilled) if teus_fulfilled > 0 else 0.0

        rows.append({
            'penalty': float(penalty),
            'status': 'ok' if np.isfinite(total_cost) else 'infeasible_or_failed',
            'teus_input': teus_in,
            'teus_fulfilled': teus_fulfilled,
            'fulfillment_pct': fulfillment_pct,
            'teus_delayed': teus_delayed,
            'delayed_pct_of_fulfilled': delayed_pct,
            'avg_delay_days': avg_delay_days,
            'total_cost': float(total_cost) if np.isfinite(total_cost) else np.nan,
            'solve_time_sec': elapsed,
            'solver_status': solver_status,
            'solver_mip_gap': solver_mip_gap,
            'solver_best_bound': solver_best_bound,
            'solver_obj': solver_obj,
            'solver_solve_time': solver_time,
            'solver_node_count': solver_nodes,
        })

        print(
            f"penalty={penalty:>10.2e} | fulfill={teus_fulfilled:>8.0f}/{teus_in:>8.0f} "
            f"({fulfillment_pct:5.1f}%) | delayed={teus_delayed:>7.0f} "
            f"| gap={solver_mip_gap if solver_mip_gap is not None else np.nan:.4f} | time={elapsed:6.1f}s"
        )

    except Exception as e:
        elapsed = time.time() - t0
        rows.append({
            'penalty': float(penalty),
            'status': f'error: {str(e)}',
            'teus_input': np.nan,
            'teus_fulfilled': np.nan,
            'fulfillment_pct': np.nan,
            'teus_delayed': np.nan,
            'delayed_pct_of_fulfilled': np.nan,
            'avg_delay_days': np.nan,
            'total_cost': np.nan,
            'solve_time_sec': elapsed,
            'solver_status': np.nan,
            'solver_mip_gap': np.nan,
            'solver_best_bound': np.nan,
            'solver_obj': np.nan,
            'solver_solve_time': np.nan,
            'solver_node_count': np.nan,
        })
        print(f"penalty={penalty:>10.2e} | ERROR after {elapsed:6.1f}s -> {e}")

# 4) Results summary
df_results = pd.DataFrame(rows)
df_results = df_results.sort_values(by='fulfillment_pct', ascending=False, na_position='last').reset_index(drop=True)

print("\n=== Grid Search Results (sorted by fulfillment_pct) ===")
print(df_results[['penalty', 'status', 'solver_status', 'fulfillment_pct', 'teus_fulfilled', 'teus_input', 'teus_delayed', 'avg_delay_days', 'solver_mip_gap', 'solver_best_bound', 'solver_obj', 'solver_solve_time', 'solver_node_count', 'solve_time_sec']])

valid = df_results[df_results['status'] == 'ok'].copy()
if len(valid) > 0:
    best = valid.sort_values(by=['fulfillment_pct', 'teus_fulfilled'], ascending=[False, False]).iloc[0]
    print("\n=== Best Penalty Candidate ===")
    print(f"Penalty:           {best['penalty']:.2e}")
    print(f"Fulfillment:       {best['fulfillment_pct']:.2f}%")
    print(f"TEUs Fulfilled:    {best['teus_fulfilled']:.0f} / {best['teus_input']:.0f}")
    print(f"TEUs Delayed:      {best['teus_delayed']:.0f}")
    print(f"Avg Delay (days):  {best['avg_delay_days']:.2f}")
    print(f"Solver Status:     {best['solver_status']}")
    print(f"MIP Gap:           {best['solver_mip_gap']}")
    print(f"Best Bound:        {best['solver_best_bound']}")
    print(f"Solver Obj:        {best['solver_obj']}")

    target80 = valid[valid['fulfillment_pct'] >= 80.0]
    if len(target80) > 0:
        print("\nFound penalties achieving >= 80% fulfillment:")
        print(target80[['penalty', 'fulfillment_pct', 'teus_fulfilled', 'teus_input', 'solver_mip_gap', 'solver_status', 'solve_time_sec']].sort_values(by='fulfillment_pct', ascending=False))
    else:
        print("\nNo grid point reached >= 80% fulfillment. Try widening grid upward and/or relaxing other competing penalties.")
else:
    print("\nNo successful solve in this grid.")

# Optional: keep result table in variable for later use
grid_search_results_unfulfilled_penalty = df_results

Loading full dataset context...
Prepared full problem: 31 lines, 741 positive-demand OD pairs

Starting grid search on unfulfilled_demand_penalty...


c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 63 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 64 times so far.

  warnings.warn(msg, UserWarning)
c:\U

penalty=  1.00e+06 | fulfill=   70850/   70850 (100.0%) | delayed=   4414 | gap=0.0976 | time= 539.6s


c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 94 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 95 times so far.

  warnings.warn(msg, UserWarning)
c:\U

penalty=  3.00e+06 | fulfill=   70850/   70850 (100.0%) | delayed=   4366 | gap=0.0999 | time= 546.9s


...

penalty=  1.00e+08 | fulfill=   70850/   70850 (100.0%) | delayed=   4481 | gap=0.0993 | time= 463.5s


c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 218 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 219 times so far.

  warnings.warn(msg, UserWarning)
c:

penalty=  3.00e+08 | fulfill=   70850/   70850 (100.0%) | delayed=   4449 | gap=0.0993 | time= 483.0s


c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 249 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 250 times so far.

  warnings.warn(msg, UserWarning)
c:

penalty=  1.00e+09 | fulfill=   70850/   70850 (100.0%) | delayed=   4443 | gap=0.0995 | time= 519.6s

=== Grid Search Results (sorted by fulfillment_pct) ===
        penalty status solver_status  fulfillment_pct  teus_fulfilled  teus_input  teus_delayed  avg_delay_days  solver_mip_gap  \
0  1.000000e+06     ok       optimal            100.0         70850.0     70850.0       4414.00        7.894222        0.097565   
1  3.000000e+06     ok       optimal            100.0         70850.0     70850.0       4365.50        7.976913        0.099928   
2  1.000000e+07     ok       optimal            100.0         70850.0     70850.0       4356.00        7.992002        0.098972   
3  1.000000e+08     ok       optimal            100.0         70850.0     70850.0       4480.75        7.791283        0.099308   
4  3.000000e+08     ok       optimal            100.0         70850.0     70850.0       4448.95        7.837414        0.099343   
5  1.000000e+09     ok       optimal            100.0  